# Two-bunch console

The calls for a session at the machine. Cells only call `twobunch_deploy` functions; every call saves its record under `data/<date>_<stage>/`. Only the v2 sections write to the machine, through `actuate.py` after `arm()` (`tests/test_read_only.py`).

Tick each checklist in the cell, and copy the completed lists into the lab-notebook commissioning page with the session's records.

## Checklists

### Read-only release (per release)
- [ ] `pixi run test` passes locally and CI passes on the release commit
- [ ] `tests/test_read_only.py` passes: no write call in the package or the notebooks, PV names only in `pvs.py`
- [ ] the step readings equal the v7 replay of the DAQ runs, and the processing equals Yiheng's (the soft-IOC and DAQ tests ran, not skipped)
- [ ] every line of the release read
- [ ] the release's code hash (CI's `pixi run code-hash`) noted here: ...

### Before each session
- [ ] `check_env()`: code hash equals the release's; no import errors
- [ ] `check_pvs()`: all PVs answer
- [ ] `geometry_test()`: camera ROI and orientation as expected (`differs` understood)
- [ ] `optics_test()`: each supply within ±1% of the DAQ, or the difference noted
- [ ] a background: taken with no beam, a pedestal, or a saved one
- [ ] `peek()`: the cut and centroids where expected
- [ ] TCAV voltage 22–30 MV and the readbacks separate off / + / − (`tcav_test()`)
- [ ] streak calibration from ops for this session: right sign, within ±20% of the expected value; entered below

### Measurement commissioning (once per setup)
- [ ] capture fraction and pulse-ID matching (`capture_test()`, `tcav_test()`)
- [ ] drop length after each flip
- [ ] whether the off reference can be reused between steps
- [ ] per-image and per-step noise against the env model
- [ ] drift per step against the trained drift
- [ ] readings inside the training range
- [ ] timing cycle chosen

## Session start

In [ ]:
from twobunch_deploy import session

session.check_env()

In [ ]:
session.check_pvs()

In [ ]:
session.geometry_test()

In [ ]:
session.optics_test()

In [ ]:
session.capture_test(seconds=10)

## Background

Run one: a background with no beam on DTOTR2, a pedestal with beam on, or a saved one (its `.npz`).

In [ ]:
background = session.take_background()
# background = session.pedestal()
# background = session.load_background("data/2026-11-04_S3/background_143016.npz")

## Looking at the frames

`peek()` is the newest frame; `peek_strip()` a few seconds of frames; `watch()` a `peek()` about once a second for 30 s (it holds the kernel: only between steps).

In [ ]:
session.peek(background)

In [ ]:
session.peek_strip(background, seconds=4)

In [ ]:
session.watch(background, seconds=30)

## The machine's PVs

In [ ]:
session.tcav_test(seconds=10)

In [ ]:
session.phases_test(seconds=60)

In [ ]:
session.drift_test(seconds=60)

## Steps

The streak calibration (µm on the screen per degree, at the live pixel size) comes from ops for this session: enter it before the first step.

In [ ]:
STREAK_UM_PER_DEG = None

In [ ]:
r = session.step(background, streak_um_per_deg=STREAK_UM_PER_DEG, seconds=15)
r.record["reading"], r.record["checks"]

In [ ]:
r.show()

In [ ]:
session.status()

## Writes (v2): armed sessions only

### Before arming
- [ ] ops agree to the session; who is at the console
- [ ] the `PREQ` feedback (BC11 bunch length) and the L2 bunch-length feedback off; the energy feedbacks on, so the LEMs stay on; all states recorded
- [ ] optics within ±1% of training (`optics_test()`)
- [ ] streak calibration from ops
- [ ] budget and limits read back from `arm()`

### End of session
- [ ] `undo_all()` or keep, stated in the e-log
- [ ] if keeping: ops reset each loop's setpoint to the current reading before re-enabling it
- [ ] feedbacks restored as agreed; `disarm()`; records copied back

In [ ]:
from twobunch_deploy import actuate

actuate.arm("Ryan")

In [ ]:
actuate.state()

In [ ]:
actuate.undo_all()

In [ ]:
actuate.disarm()

## Policy (v2): goals, shadow mode, approved cycles

The policy proposes L1 and L2 moves from each valid reading (`advise.py`). Use `advise.step()` here instead of `session.step()`: it remembers the readings for goals and proposals. Unarmed, `cycle()` is shadow mode: it proposes and logs, and writes nothing. Armed, it asks `apply? [y/N]` after each proposal: `y` applies it if answered within 60 s; anything else declines. An invalid reading gets no proposal. Manual moves are refused while an episode runs. Each episode's proposals and decisions go in `data/<date>_V2/episode_<HHMMSS_mmm>.jsonl`.

1. **Goal:** with the beam where ops want it, three valid steps, then `record_goal`.
2. **Shadow:** `use_goal`, then cycles, unarmed.
3. **Approved:** `end_episode()`; arm (checklist above); detune with `move_l1` / `move_l2`, one step at a time inside the budget; `use_goal`; then cycles.
4. **End:** `end_episode()`, then the end-of-session checklist above.

In [ ]:
from twobunch_deploy import advise

for _ in range(3):
    r = advise.step(background, streak_um_per_deg=STREAK_UM_PER_DEG, seconds=15)
    print(r.record["valid"], r.record["reading"])

In [ ]:
advise.record_goal("A")

In [ ]:
actuate.move_l1(0.25)

In [ ]:
actuate.move_l2(0.3)

In [ ]:
advise.use_goal("A")

In [ ]:
for _ in range(10):
    advise.cycle(background, streak_um_per_deg=STREAK_UM_PER_DEG, seconds=15)

In [ ]:
advise.state()

In [ ]:
advise.end_episode()